# 02b — The Perceptron Learning Rule

This notebook implements the Perceptron from scratch in pure Python.

The Perceptron (Rosenblatt, 1958) extends the McCulloch-Pitts neuron by making weights **learnable from data**.

**What we build:**
- The perceptron model (forward pass)
- The perceptron learning rule
- Full training trace on OR — epoch by epoch
- XOR failure — the perceptron cannot solve it, and we show exactly why

---
## Part 1: The Perceptron Model

```
x₁ ──(w₁)──┐
            ├──► z = w₁x₁ + w₂x₂ + b ──► f(z) ──► ŷ ∈ {0,1}
x₂ ──(w₂)──┘
```

Step function: ŷ = 1 if z ≥ threshold, else 0

Unlike the MP neuron, weights w₁, w₂ and bias b are **learned from data**.

In [11]:
def step(z, threshold=0.5):
    """Step activation function."""
    return 1 if z >= threshold else 0

def forward(x, weights, bias, threshold=0.5):
    """Perceptron forward pass: compute weighted sum then apply step function."""
    z = sum(w * xi for w, xi in zip(weights, x)) + bias
    return step(z, threshold), z

print('Perceptron model defined.')
print()
print('Forward pass: z = w1*x1 + w2*x2 + b')
print('Output:       ŷ = 1 if z >= 0.5, else 0')
print()
print('Test with w1=0.5, w2=0.5, b=0.0, input=[1,1]:')
pred, z = forward([1,1], [0.5, 0.5], 0.0)
print(f'  z = {z},  ŷ = {pred}')

Perceptron model defined.

Forward pass: z = w1*x1 + w2*x2 + b
Output:       ŷ = 1 if z >= 0.5, else 0

Test with w1=0.5, w2=0.5, b=0.0, input=[1,1]:
  z = 1.0,  ŷ = 1


---
## Part 2: The Perceptron Learning Rule

For each training sample:
1. Compute prediction ŷ
2. Compute error e = y − ŷ
3. Update: wᵢ ← wᵢ + η × e × xᵢ
4. Update: b  ← b  + η × e

If prediction is correct (e = 0): no update.
If prediction is wrong: weights shift toward the correct answer.

In [12]:
def perceptron_update(x, y, weights, bias, lr, threshold=0.5):
    """
    One perceptron learning step.
    Returns updated weights, bias, error, and prediction.
    """
    y_hat, z = forward(x, weights, bias, threshold)
    error = y - y_hat
    new_weights = [w + lr * error * xi for w, xi in zip(weights, x)]
    new_bias    = bias + lr * error
    return new_weights, new_bias, error, y_hat

print('Learning rule defined.')
print()
print('e = y - ŷ')
print('  e =  0: correct prediction → no update')
print('  e = +1: predicted 0, true is 1 → weights increase')
print('  e = -1: predicted 1, true is 0 → weights decrease')

Learning rule defined.

e = y - ŷ
  e =  0: correct prediction → no update
  e = +1: predicted 0, true is 1 → weights increase
  e = -1: predicted 1, true is 0 → weights decrease


---
## Part 3: Training on OR — Full Epoch-by-Epoch Trace

OR truth table:
```
x1  x2  y
 0   0  0
 0   1  1
 1   0  1
 1   1  1
```

Starting from zero weights. We trace every update.

In [13]:
# OR training data
or_data = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 1),
]

# Initial parameters
weights   = [0.0, 0.0]
bias      = 0.0
lr        = 0.1
threshold = 0.5

print(f'Initial: w1={weights[0]}, w2={weights[1]}, b={bias}')
print()

for epoch in range(1, 11):
    total_errors = 0
    print(f'Epoch {epoch}')
    print('-' * 65)

    for x, y in or_data:
        weights, bias, error, y_hat = perceptron_update(
            x, y, weights, bias, lr, threshold
        )
        total_errors += abs(error)
        update_str = 'updated' if error != 0 else 'no change'
        print(f'  x={x}, y={y}, ŷ={y_hat}, e={error:+.0f}  '
              f'→ w1={weights[0]:.2f}, w2={weights[1]:.2f}, b={bias:.2f}  [{update_str}]')

    print(f'  End of epoch {epoch}: total errors = {total_errors}')
    print()

    if total_errors == 0:
        print(f'Converged at epoch {epoch} — no errors on full pass.')
        break

Initial: w1=0.0, w2=0.0, b=0.0

Epoch 1
-----------------------------------------------------------------
  x=[0, 0], y=0, ŷ=0, e=+0  → w1=0.00, w2=0.00, b=0.00  [no change]
  x=[0, 1], y=1, ŷ=0, e=+1  → w1=0.00, w2=0.10, b=0.10  [updated]
  x=[1, 0], y=1, ŷ=0, e=+1  → w1=0.10, w2=0.10, b=0.20  [updated]
  x=[1, 1], y=1, ŷ=0, e=+1  → w1=0.20, w2=0.20, b=0.30  [updated]
  End of epoch 1: total errors = 3

Epoch 2
-----------------------------------------------------------------
  x=[0, 0], y=0, ŷ=0, e=+0  → w1=0.20, w2=0.20, b=0.30  [no change]
  x=[0, 1], y=1, ŷ=1, e=+0  → w1=0.20, w2=0.20, b=0.30  [no change]
  x=[1, 0], y=1, ŷ=1, e=+0  → w1=0.20, w2=0.20, b=0.30  [no change]
  x=[1, 1], y=1, ŷ=1, e=+0  → w1=0.20, w2=0.20, b=0.30  [no change]
  End of epoch 2: total errors = 0

Converged at epoch 2 — no errors on full pass.


In [14]:
# Final verification
print('Final weights: w1={:.2f}, w2={:.2f}, b={:.2f}'.format(weights[0], weights[1], bias))
print()
print('Final predictions on all OR inputs:')
print('-' * 40)
print(f'  {"x":>8}  {"y":>4}  {"ŷ":>4}  {"z":>6}  {"correct?":>9}')

for x, y in or_data:
    y_hat, z = forward(x, weights, bias, threshold)
    check = '✓' if y_hat == y else '✗'
    print(f'  {str(x):>8}  {y:>4}  {y_hat:>4}  {z:>6.3f}  {check:>9}')

print()
print('Decision boundary: w1*x1 + w2*x2 + b = 0.5')
print(f'  {weights[0]:.2f}*x1 + {weights[1]:.2f}*x2 + {bias:.2f} = 0.5')

Final weights: w1=0.20, w2=0.20, b=0.30

Final predictions on all OR inputs:
----------------------------------------
         x     y     ŷ       z   correct?
    [0, 0]     0     0   0.300          ✓
    [0, 1]     1     1   0.500          ✓
    [1, 0]     1     1   0.500          ✓
    [1, 1]     1     1   0.700          ✓

Decision boundary: w1*x1 + w2*x2 + b = 0.5
  0.20*x1 + 0.20*x2 + 0.30 = 0.5


---
## Part 4: XOR — The Perceptron Fails

XOR truth table:
```
x1  x2  y
 0   0  0
 0   1  1
 1   0  1
 1   1  0
```

XOR is not linearly separable. No single straight line can separate class 1 from class 0.
The perceptron will never converge — it will keep making errors forever.

In [15]:
# XOR training data
xor_data = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]

# Reset weights
weights_xor = [0.0, 0.0]
bias_xor    = 0.0

print('Training perceptron on XOR for 20 epochs...')
print()
print(f'  {"Epoch":>6}  {"Errors":>8}  {"w1":>8}  {"w2":>8}  {"b":>8}')

for epoch in range(1, 21):
    total_errors = 0
    for x, y in xor_data:
        weights_xor, bias_xor, error, _ = perceptron_update(
            x, y, weights_xor, bias_xor, lr=0.1, threshold=0.5
        )
        total_errors += abs(error)
    print(f'  {epoch:>6}  {total_errors:>8}  {weights_xor[0]:>8.3f}  {weights_xor[1]:>8.3f}  {bias_xor:>8.3f}')

print()
print('Errors never reach 0 — the perceptron cannot solve XOR.')

Training perceptron on XOR for 20 epochs...

   Epoch    Errors        w1        w2         b
       1         2     0.100     0.100     0.200
       2         3     0.100     0.100     0.300
       3         2     0.000     0.100     0.300
       4         3     0.000     0.100     0.400
       5         2     0.000     0.000     0.400
       6         2    -0.100     0.000     0.400
       7         3    -0.100     0.000     0.500
       8         4    -0.100     0.000     0.500
       9         4    -0.100     0.000     0.500
      10         4    -0.100     0.000     0.500
      11         4    -0.100     0.000     0.500
      12         4    -0.100     0.000     0.500
      13         4    -0.100     0.000     0.500
      14         4    -0.100     0.000     0.500
      15         4    -0.100     0.000     0.500
      16         4    -0.100     0.000     0.500
      17         4    -0.100     0.000     0.500
      18         4    -0.100     0.000     0.500
      19         4    -0

In [16]:
# Show the algebraic contradiction
print('Why XOR is impossible for a single perceptron:')
print()
print('For a perceptron to solve XOR, we need weights w1, w2, b such that:')
print()
print('  (0,0) → 0:  w1*0 + w2*0 + b < 0.5   →  b < 0.5          ... (i)')
print('  (0,1) → 1:  w1*0 + w2*1 + b >= 0.5  →  w2 + b >= 0.5    ... (ii)')
print('  (1,0) → 1:  w1*1 + w2*0 + b >= 0.5  →  w1 + b >= 0.5    ... (iii)')
print('  (1,1) → 0:  w1*1 + w2*1 + b < 0.5   →  w1 + w2 + b < 0.5 ... (iv)')
print()
print('From (ii): w2 >= 0.5 - b > 0  (using i: b < 0.5)')
print('From (iii): w1 >= 0.5 - b > 0')
print()
print('Adding (ii) and (iii):')
print('  w1 + w2 + 2b >= 1.0')
print('  w1 + w2 + b  >= 1.0 - b > 0.5  (since b < 0.5)')
print()
print('But (iv) requires: w1 + w2 + b < 0.5')
print()
print('CONTRADICTION — no values of w1, w2, b can satisfy all four conditions.')
print()
print('Solution: use multiple layers → Multilayer Perceptron (Note 03).')

Why XOR is impossible for a single perceptron:

For a perceptron to solve XOR, we need weights w1, w2, b such that:

  (0,0) → 0:  w1*0 + w2*0 + b < 0.5   →  b < 0.5          ... (i)
  (0,1) → 1:  w1*0 + w2*1 + b >= 0.5  →  w2 + b >= 0.5    ... (ii)
  (1,0) → 1:  w1*1 + w2*0 + b >= 0.5  →  w1 + b >= 0.5    ... (iii)
  (1,1) → 0:  w1*1 + w2*1 + b < 0.5   →  w1 + w2 + b < 0.5 ... (iv)

From (ii): w2 >= 0.5 - b > 0  (using i: b < 0.5)
From (iii): w1 >= 0.5 - b > 0

Adding (ii) and (iii):
  w1 + w2 + 2b >= 1.0
  w1 + w2 + b  >= 1.0 - b > 0.5  (since b < 0.5)

But (iv) requires: w1 + w2 + b < 0.5

CONTRADICTION — no values of w1, w2, b can satisfy all four conditions.

Solution: use multiple layers → Multilayer Perceptron (Note 03).


In [17]:
# Visualise the problem geometrically (text-based)
print('XOR in the input space:')
print()
print('  x2')
print('   1 |  ● (0,1)     ○ (1,1)')
print('     |')
print('   0 |  ○ (0,0)     ● (1,0)')
print('     +------------------  x1')
print('        0           1')
print()
print('  ● = XOR = 1 (class 1)')
print('  ○ = XOR = 0 (class 0)')
print()
print('Try to draw ONE straight line that separates ● from ○.')
print('It is impossible — the classes are not linearly separable.')
print()
print('OR in the input space (for comparison):')
print()
print('  x2')
print('   1 |  ● (0,1)     ● (1,1)')
print('     |         \\')
print('   0 |  ○ (0,0)  \\  ● (1,0)')
print('     +------------------  x1')
print('        0           1')
print()
print('OR: one straight line separates ○ from ● — linearly separable. ✓')

XOR in the input space:

  x2
   1 |  ● (0,1)     ○ (1,1)
     |
   0 |  ○ (0,0)     ● (1,0)
     +------------------  x1
        0           1

  ● = XOR = 1 (class 1)
  ○ = XOR = 0 (class 0)

Try to draw ONE straight line that separates ● from ○.
It is impossible — the classes are not linearly separable.

OR in the input space (for comparison):

  x2
   1 |  ● (0,1)     ● (1,1)
     |         \
   0 |  ○ (0,0)  \  ● (1,0)
     +------------------  x1
        0           1

OR: one straight line separates ○ from ● — linearly separable. ✓


---
## Summary

| | McCulloch-Pitts | Perceptron |
|---|---|---|
| Weights | Fixed by designer | Learned from data |
| Learning rule | None | w ← w + η(y−ŷ)x |
| Can solve OR | Yes (manually) | Yes (automatically) |
| Can solve XOR | No | No |
| Why XOR fails | Not linearly separable | Not linearly separable |

**The perceptron learning rule guarantees convergence if and only if the data is linearly separable.**

XOR is not linearly separable → the perceptron will never converge on XOR.

The solution is to stack multiple layers — the Multilayer Perceptron (Note 03).